In [1]:
!pip install transformers datasets evaluate captum scikit-learn pandas -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


# Attribution Analysis
IG attributions vs human rationales across all tokenizers.


In [2]:
import pandas as pd
import numpy as np
import time
import sys
import os
sys.path.append(os.path.abspath('../src'))
from attribution import LRAttribution, RobertaAttribution, BertAttribution, AlbertAttribution
from evaluation import aggregate_word_attributions, compute_token_f1
from transformers import AutoTokenizer


In [3]:
adv_df = pd.read_pickle('../data/processed/adv_test.pkl')
row = adv_df.iloc[0]
original_text = row['original_text']
perturbed_text = row['perturbed_text']


In [4]:
attributions = {
    'lr_char': LRAttribution(model_dir='../models/lr_char'),
    'lr_word': LRAttribution(model_dir='../models/lr_word'),
    'roberta': RobertaAttribution(model_dir='../models/roberta'),
    'bert': BertAttribution(model_dir='../models/bert'),
    'albert': AlbertAttribution(model_dir='../models/albert')
}


Model loaded.


Model loaded.


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

In [5]:
for name, attr_model in attributions.items():
    orig_tokens, orig_scores = attr_model.get_attribution(original_text)
    adv_tokens, adv_scores = attr_model.get_attribution(perturbed_text)
    print(f'\n--- {name.upper()} ---')
    print('Original:', orig_tokens)
    print('Adversarial:', adv_tokens)



--- LR_CHAR ---
Original: ['i', 'retrospect', 'it', 'pretty', 'obvious', 'to', 'see', 'that', 'obama', 'is', 'pro', 'muslim', 'if', 'not', 'a', 'muslim', 'himself', 'he', 'stabbed', 'israel', 'in', 'the', 'back', 'tried', 'to', 'give', 'millions', 'to', 'the', 'palestinians', 'and', 'literally', 'gave', 'iran', 'pallets', 'full', 'of', 'money', 'to', 'help', 'them', 'further', 'there', 'nuclear', 'and', 'missile', 'program']
Adversarial: ['i', 'retrospect', 'it', 'p.r.e.t.t.y', 'obvious', 'to', 's.e.e', 't.h.a.t', 'obama', 'is', 'pro', 'm.u.s.l.i.m', 'if', 'not', 'a', 'muslim', 'himself', 'he', 'stabbed', 'israel', 'in', 'the', 'back', 't.r.i.e.d', 'to', 'give', 'm.i.l.l.i.o.n.s', 'to', 'the', 'palestinians', 'and', 'l.i.t.e.r.a.l.l.y', 'gave', 'i.r.a.n', 'pallets', 'full', 'of', 'money', 'to', 'help', 't.h.e.m', 'further', 'there', 'nuclear', 'and', 'missile', 'p.r.o.g.r.a.m']

--- LR_WORD ---
Original: ['i', 'retrospect', 'it', 'pretty', 'obvious', 'to', 'see', 'that', 'obama', 'is'


--- ROBERTA ---
Original: ['<s>', 'i', 'Ġretrospect', 'Ġit', 'Ġpretty', 'Ġobvious', 'Ġto', 'Ġsee', 'Ġthat', 'Ġob', 'ama', 'Ġis', 'Ġpro', 'Ġmus', 'lim', 'Ġif', 'Ġnot', 'Ġa', 'Ġmus', 'lim', 'Ġhimself', 'Ġhe', 'Ġstabbed', 'Ġis', 'rael', 'Ġin', 'Ġthe', 'Ġback', 'Ġtried', 'Ġto', 'Ġgive', 'Ġmillions', 'Ġto', 'Ġthe', 'Ġpal', 'est', 'in', 'ians', 'Ġand', 'Ġliterally', 'Ġgave', 'Ġir', 'an', 'Ġpal', 'lets', 'Ġfull', 'Ġof', 'Ġmoney', 'Ġto', 'Ġhelp', 'Ġthem', 'Ġfurther', 'Ġthere', 'Ġnuclear', 'Ġand', 'Ġmissile', 'Ġprogram', '</s>']
Adversarial: ['<s>', 'i', 'Ġretrospect', 'Ġit', 'Ġp', '.', 'r', '.', 'e', '.', 't', '.', 't', '.', 'y', 'Ġobvious', 'Ġto', 'Ġs', '.', 'e', '.', 'e', 'Ġt', '.', 'h', '.', 'a', '.', 't', 'Ġob', 'ama', 'Ġis', 'Ġpro', 'Ġm', '.', 'u', '.', 's', '.', 'l', '.', 'i', '.', 'm', 'Ġif', 'Ġnot', 'Ġa', 'Ġmus', 'lim', 'Ġhimself', 'Ġhe', 'Ġstabbed', 'Ġis', 'rael', 'Ġin', 'Ġthe', 'Ġback', 'Ġt', '.', 'r', '.', 'i', '.', 'e', '.', 'd', 'Ġto', 'Ġgive', 'Ġm', '.', 'i', '.', 'l', '.', 'l',


--- BERT ---
Original: ['[CLS]', 'i', 'retro', '##sp', '##ect', 'it', 'pretty', 'obvious', 'to', 'see', 'that', 'obama', 'is', 'pro', 'muslim', 'if', 'not', 'a', 'muslim', 'himself', 'he', 'stabbed', 'israel', 'in', 'the', 'back', 'tried', 'to', 'give', 'millions', 'to', 'the', 'palestinians', 'and', 'literally', 'gave', 'iran', 'pal', '##lets', 'full', 'of', 'money', 'to', 'help', 'them', 'further', 'there', 'nuclear', 'and', 'missile', 'program', '[SEP]']
Adversarial: ['[CLS]', 'i', 'retro', '##sp', '##ect', 'it', 'p', '.', 'r', '.', 'e', '.', 't', '.', 't', '.', 'y', 'obvious', 'to', 's', '.', 'e', '.', 'e', 't', '.', 'h', '.', 'a', '.', 't', 'obama', 'is', 'pro', 'm', '.', 'u', '.', 's', '.', 'l', '.', 'i', '.', 'm', 'if', 'not', 'a', 'muslim', 'himself', 'he', 'stabbed', 'israel', 'in', 'the', 'back', 't', '.', 'r', '.', 'i', '.', 'e', '.', 'd', 'to', 'give', 'm', '.', 'i', '.', 'l', '.', 'l', '.', 'i', '.', 'o', '.', 'n', '.', 's', 'to', 'the', 'palestinians', 'and', 'l', '.', '


--- ALBERT ---
Original: ['[CLS]', '▁i', '▁retro', 'spect', '▁it', '▁pretty', '▁obvious', '▁to', '▁see', '▁that', '▁obama', '▁is', '▁pro', '▁muslim', '▁if', '▁not', '▁a', '▁muslim', '▁himself', '▁he', '▁stabbed', '▁israel', '▁in', '▁the', '▁back', '▁tried', '▁to', '▁give', '▁millions', '▁to', '▁the', '▁palestinian', 's', '▁and', '▁literally', '▁gave', '▁iran', '▁pallet', 's', '▁full', '▁of', '▁money', '▁to', '▁help', '▁them', '▁further', '▁there', '▁nuclear', '▁and', '▁missile', '▁program', '[SEP]']
Adversarial: ['[CLS]', '▁i', '▁retro', 'spect', '▁it', '▁p', '.', 'r', '.', 'e', '.', 't', '.', 't', '.', 'y', '▁obvious', '▁to', '▁', 's', '.', 'e', '.', 'e', '▁', 't', '.', 'h', '.', 'a', '.', 't', '▁obama', '▁is', '▁pro', '▁m', '.', 'u', '.', 's', '.', 'l', '.', 'i', '.', 'm', '▁if', '▁not', '▁a', '▁muslim', '▁himself', '▁he', '▁stabbed', '▁israel', '▁in', '▁the', '▁back', '▁', 't', '.', 'r', '.', 'i', '.', 'e', '.', 'd', '▁to', '▁give', '▁m', '.', 'i', '.', 'l', '.', 'l', '.', 'i', '.'

# Token-Level F1
Compare IG attribution alignment with human rationales before and after adversarial attack.
space_injection is excluded from Token-F1 because it breaks word-level rationale alignment.


In [6]:
test_df = pd.read_pickle('../data/processed/test.pkl')
toxic_test = test_df[(test_df['label'] == 1) & (test_df['rationale'].apply(lambda r: sum(r) > 0))].copy()
print(f'Toxic samples with rationales: {len(toxic_test)}')


Toxic samples with rationales: 822


In [7]:
model_configs = {
    'RoBERTa': (RobertaAttribution, '../models/roberta'),
    'BERT': (BertAttribution, '../models/bert'),
    'ALBERT': (AlbertAttribution, '../models/albert'),
}

all_results = {}


In [8]:
for model_name, (attr_class, model_dir) in model_configs.items():
    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f'{"="*60}')
    
    attr_model = attr_class(model_dir=model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    
    orig_results = []
    adv_results = {a: [] for a in ['char_insertion', 'leetspeak', 'homoglyph_swap']}
    
    start = time.time()
    
    for idx, (_, row) in enumerate(toxic_test.iterrows()):
        if idx % 50 == 0:
            print(f'  {idx}/{len(toxic_test)} ({time.time()-start:.0f}s)')
        
        try:
            _, scores = attr_model.get_attribution(row['text'], target_class=1)
            word_attrs = aggregate_word_attributions(tokenizer, row['text'], scores)
            r = compute_token_f1(word_attrs, row['rationale'])
            if r: orig_results.append(r)
        except:
            continue
        
        for _, adv_row in adv_df[adv_df['post_id'] == row['post_id']].iterrows():
            if adv_row['attack_type'] == 'space_injection':
                continue
            try:
                _, adv_scores = attr_model.get_attribution(adv_row['perturbed_text'], target_class=1)
                adv_word_attrs = aggregate_word_attributions(tokenizer, adv_row['perturbed_text'], adv_scores)
                r = compute_token_f1(adv_word_attrs, row['rationale'])
                if r: adv_results[adv_row['attack_type']].append(r)
            except:
                continue
    
    print(f'  Done: {len(orig_results)} orig + {sum(len(v) for v in adv_results.values())} adv ({time.time()-start:.0f}s)')
    all_results[model_name] = {'original': orig_results, 'adversarial': adv_results}



RoBERTa


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0/822 (0s)


  50/822 (61s)


  100/822 (123s)


...

  650/822 (1029s)


  700/822 (1101s)


  750/822 (1178s)


  800/822 (1249s)


  Done: 822 orig + 2466 adv (1289s)


## Results


In [9]:
for model_name in ['RoBERTa', 'BERT', 'ALBERT']:
    data = all_results[model_name]
    orig = data['original']
    
    orig_f1 = np.mean([r['f1'] for r in orig])
    orig_p = np.mean([r['precision'] for r in orig])
    orig_r = np.mean([r['recall'] for r in orig])
    
    all_adv = [r for v in data['adversarial'].values() for r in v]
    adv_f1 = np.mean([r['f1'] for r in all_adv])
    adv_p = np.mean([r['precision'] for r in all_adv])
    adv_r = np.mean([r['recall'] for r in all_adv])
    
    print(f'\n{model_name}')
    print(f'  Original  - P: {orig_p:.3f}  R: {orig_r:.3f}  F1: {orig_f1:.3f}  (n={len(orig)})')
    print(f'  Adversarial - P: {adv_p:.3f}  R: {adv_r:.3f}  F1: {adv_f1:.3f}  (n={len(all_adv)})')
    print(f'  Delta F1: {adv_f1 - orig_f1:+.3f}')
    
    for attack in ['char_insertion', 'leetspeak', 'homoglyph_swap']:
        results = data['adversarial'][attack]
        if results:
            print(f'    {attack}: F1={np.mean([r["f1"] for r in results]):.3f} (n={len(results)})')



RoBERTa
  Original  - P: 0.128  R: 0.128  F1: 0.128  (n=822)
  Adversarial - P: 0.170  R: 0.170  F1: 0.170  (n=2466)
  Delta F1: +0.041
    char_insertion: F1=0.224 (n=822)
    leetspeak: F1=0.155 (n=822)
    homoglyph_swap: F1=0.130 (n=822)

BERT
  Original  - P: 0.122  R: 0.122  F1: 0.122  (n=822)
  Adversarial - P: 0.175  R: 0.175  F1: 0.175  (n=2466)
  Delta F1: +0.053
    char_insertion: F1=0.223 (n=822)
    leetspeak: F1=0.126 (n=822)
    homoglyph_swap: F1=0.177 (n=822)

ALBERT
  Original  - P: 0.197  R: 0.197  F1: 0.197  (n=822)
  Adversarial - P: 0.178  R: 0.178  F1: 0.178  (n=2466)
  Delta F1: -0.019
    char_insertion: F1=0.159 (n=822)
    leetspeak: F1=0.190 (n=822)
    homoglyph_swap: F1=0.184 (n=822)


In [10]:
import json
save_data = {}
for m, data in all_results.items():
    save_data[m] = {'original': data['original'], 'adversarial': data['adversarial']}
with open('../results/token_f1_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print('Saved to results/token_f1_results.json')


Saved to results/token_f1_results.json
